<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
import numpy as np
from functools import lru_cache

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

@lru_cache(maxsize=16)
def _cached_split(seed):
    X, y = fetch_openml("Fashion-MNIST", version=1, as_frame=False, return_X_y=True)
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )
    return X_train, X_test, y_train, y_test

def load_data(seed=42):
    X_train, X_test, y_train, y_test = _cached_split(seed)
    return X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy()

X_train, X_test, y_train, y_test = load_data(seed=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print("Classes:", np.unique(y_train))

Não é preciso normalizar os dados pois, em modelos compostos de/por arvóres de decisão, a(s) árvore(s) divide(m) os dados na base dos atributos e não da distância das amostras ou da escala delas como em outros tipos de modelos (Redes neurais, KNN, etc).

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
def train_random_forest(X_train, y_train, seed=42, max_depth_forest=0, forest_estimators=120):
    if (max_depth_forest == 0):
      max_depth_forest = None

    model = RandomForestClassifier(
        n_estimators=forest_estimators,
        max_depth=max_depth_forest,
        random_state=seed,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42, max_depth_boost=2, adaboost_estimators=120):
    base = DecisionTreeClassifier(max_depth=max_depth_boost, random_state=seed)

    try:
        model = AdaBoostClassifier(
            estimator=base,
            n_estimators=adaboost_estimators,
            learning_rate=0.7,
            random_state=seed
        )
    except TypeError:
        model = AdaBoostClassifier(
            base_estimator=base,
            n_estimators=adaboost_estimators,
            learning_rate=0.7,
            random_state=seed
        )

    model.fit(X_train, y_train)
    return model

X_train, X_test, y_train, y_test = load_data()

rf_model = train_random_forest(X_train, y_train)
ab_model = train_adaboost(X_train, y_train)

print(type(rf_model).__name__, type(ab_model).__name__)

A train_random_forest recebe os dados e atributos de treino, seta a seed como sendo 42 para garantir a reprodutibilidade, define qual é a profundidade máxima da árvore e o número de árvores que irão ser usadas na floresta. O modelo é treinado usando a classe RandomForestClassifier. Nela, o modelo usará 120 árvores diferentes e a aleatoriedade é colocada como a mesma da seed. Além disso, o n_jobs é colocado como -1 para que todos os cores da placa do computador sejam usados. O modelo é então treinado usando os dados e atributos de treino e a função retorna o modelo pronto.

Já no train_adaboost, ele também recebe os mesmos parâmetros que a função anterior, mas procede de forma diferente. Primeiro, uma única árvore rasa é criada (usando a aleatoriedade pré-definida de antes). Depois, o processo de boosting é aplicado 120 vezes a aquela primeira árvore (120 iterações de modelos ficando cada vez melhores do que o anterior) com uma intensidade de correção a erros de 0,7 enquanto usa a mesma seed de sempre (Como o sklearn muda a sintaxe baseado em se é uma versão antiga ou nova, o código tenta usar as duas sintaxes para ver qual delas funciona). O modelo é então treinado usando os dados e atributos de treino e a função retorna o modelo pronto.

No final, usamos a função de load_data para carregar os dados e então usamos as duas funções previamente criadas para fazer os dois modelos. Checamos então se eles foram feitos checando pelos seus nomes no terminal.

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão Macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_test, y_pred, average="macro", zero_division=0)
    }

rf_estats = evaluate(rf_model, X_test, y_test)
ab_estats = evaluate(ab_model, X_test, y_test)

print(f"Avaliação do Random Forest: {rf_estats}\n")
print(f"Avaliação do AdaBoost: {ab_estats}")

Na função evaluate, os "chutes" do modelo baseado nos dados de teste são comparados com os dados verdadeiros de teste para checar se ele acertou ou não e avaliar sua acurácia, precisão, recall e f1 score (A razão por trás de ter "macro" no nome da maioria das variáveis é que, quando houver o cálculo das estátisticas, o código fará uma média de todos os atributos de uma forma que não priorize uma classe que aparece muito mais que as outras).

Para testar isso, pegamos os dois modelos já feitos e colocamos eles nessas funções para analisar suas métricas, que então vemos no terminal.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type, seed=42, model_max_depth=0, n_estimators=120):
    max_depth_forest = model_max_depth
    max_depth_boost = model_max_depth

    forest_estimators = n_estimators
    adaboost_estimators = n_estimators

    if model_max_depth == 0:
      max_depth_forest = 0
      max_depth_boost = 2

    X_train, X_test, y_train, y_test = load_data(seed=seed)

    if model_type.lower() == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, max_depth_forest=max_depth_forest, forest_estimators=forest_estimators)
    elif model_type.lower() == "ab":
        model = train_adaboost(X_train, y_train, seed=seed, max_depth_boost=max_depth_boost, adaboost_estimators=adaboost_estimators)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'.")

    acc = evaluate(model, X_test, y_test)
    return acc

for i in range(0, 20, 5):
  print(f"Random Forest (Profundidade {i}): {run_pipeline("rf", seed=42, model_max_depth=i)}")

  print(f"AdaBoost (Profundidade {i}): {run_pipeline("ab", seed=42, model_max_depth=i)}\n")

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

O overfitting começa a partir da profundidade 20. A árvore consegue 100% no treino quando max_depth=None porque ela não tem limite de profundidade, causando no fato de ela fazer um nó exclusivo para cada dado nos piores dos casos. Isso faz com que ela memorize os dados e não aprenda as regras ou padrões por trás realmente, causando overfitting.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
for i in range(0, 20, 5):
  print(f"Random Forest (Profundidade {i}): {run_pipeline("rf", seed=42, model_max_depth=i)}")

  print(f"AdaBoost (Profundidade {i}): {run_pipeline("ab", seed=42, model_max_depth=i)}\n")

O random forest apresentou o melhor desempenho inicial porque todas as métricas foram significativamente melhores do que o AdaBoost (que melhorou comparativamente pouco nesse caso).

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
for i in [7, 19, 42, 50]:
  print(f"Random Forest (Seed: {i})): {run_pipeline("rf", seed=i)}")

  print(f"AdaBoost (Seed: {i}): {run_pipeline("ab", seed=i)}\n")

Sim, os resultados mudaram. O experimento é reprodutível porque a mesma seed sempre produz os mesmos resultados todas as vezes.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
train_rf_acc = evaluate(rf_model, X_train, y_train)
test_rf_acc = evaluate(rf_model, X_test, y_test)

train_ab_acc = evaluate(ab_model, X_train, y_train)
test_ab_acc = evaluate(ab_model, X_test, y_test)

print("RF treino:", train_rf_acc)
print("RF teste:", test_rf_acc)
print("Gap RF:", train_rf_acc - test_rf_acc)

print("\nAB treino:", train_ab_acc)
print("AB teste:", test_ab_acc)
print("Gap AB:", train_ab_acc - test_ab_acc)

Existe, pois a acurácia no conjunto de treino foi de 100% no caso do Random Forest, o que é um ótimo indicador que ocorreu overfitting. Apesar disso, o modelo que mais sofre com overfitting é o AdaBoosting pois o Random Forest suaviza os erros de suas árvores individuais enquanto que o AdaBoosting foca nos erros cometidos pelos últimos modelos (que podem ter sido devido a ruídos, aumentando a chance de overfitting).

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
for i in [30, 60, 120, 200]:
  print(f"Random Forest (Estimadores: {i})): {run_pipeline("rf", seed=42, n_estimators=i)}")

  print(f"AdaBoost (Estimadores: {i}): {run_pipeline("ab", seed=42, n_estimators=i)}\n")

Sim. A maior mudança observável ocorre no AdaBoost que cresce de forma muito grande de acordo com o número de estimadores (ao contrário do Random Forest que cresce e depois estabiliza depois de um tempo).

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acurácia pode acabar passando a impressão errada se houver várias classes diferentes e o modelo tiver sido ruim em algumas delas, por isso que é importante usar precisão, recall e f1 score para analisar o modelo por completo.

2. Mudando vários parâmetros enquanto fixa a aleatoriedade no lugar. O mesmo comportamento em vários casos com a mesma aleatoriedade garante que o resultado não ocorreu por acaso.

3. Validação cruzada não foi utilizada e os hiperparâmetros poderiam ter sido mais bem explorados

4. O pipeline é seguro em termos da estrutura geral dele (ter funções separadas com métricas claras e controle de todas as variáveis desejadas), mas pode ser melhorado com a adição de validação cruzada, matriz de confusão, etc.